<a href="https://colab.research.google.com/github/ankita-rdk/EEG-Analysis/blob/main/med_eeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive
import os
import random
import sys
from pathlib import Path
import numpy as np
import torch

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/OpenMedWave"
CACHE_DIR = "/content/openmedwave_cache"
MEMMAP_DIR = f"{CACHE_DIR}/memmap"
CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
BENCH_DIR = "/content/drive/MyDrive/NeuroWaveMed_Benchmarks"
BENCH_LATEST_DIR = f"{BENCH_DIR}/latest"

for d in [
    f"{PROJECT_DIR}/openmedwave",
    f"{PROJECT_DIR}/configs",
    MEMMAP_DIR,
    CKPT_DIR,
    BENCH_LATEST_DIR,
]:
    os.makedirs(d, exist_ok=True)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

!pip install -q mne pyyaml scipy scikit-learn gradio matplotlib pandas

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA available: False | Device: CPU


In [9]:
%%writefile configs/montages.yaml
target_sfreq: 256
window_sec: 4.0
stride_sec: 2.0
pre_probe_sec: 12.0

canonical_channels:
  - Fp1
  - Fp2
  - AF3
  - AF4
  - AF7
  - AF8
  - F3
  - F4
  - F7
  - F8
  - T7
  - T8
  - TP9
  - TP10
  - P7
  - P8
  - Pz
  - O1
  - O2

wearable_montages:
  neurosky_1ch: ["Fp1"]
  muse2_4ch: ["TP9", "AF7", "AF8", "TP10"]
  emotiv_insight_5ch: ["AF3", "AF4", "T7", "T8", "Pz"]
  openbci_8ch: ["Fp1", "Fp2", "F3", "F4", "T7", "T8", "O1", "O2"]
  emotiv_epoc_14ch: ["AF3", "F7", "F3", "T7", "P7", "O1", "O2", "P8", "T8", "F4", "F8", "AF4", "Fp1", "Fp2"]

Overwriting configs/montages.yaml


In [10]:
%%writefile openmedwave/preprocessing.py
import math
from pathlib import Path
from typing import List, Tuple
import numpy as np
import pandas as pd
from scipy import signal
import mne


def standardize_channel_names(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    alias_map = {
        "T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8",
        "FP1": "Fp1", "FP2": "Fp2", "PZ": "Pz", "FZ": "Fz", "CZ": "Cz",
    }
    rename_dict = {}
    for ch in raw.ch_names:
        clean = ch.replace("EEG ", "").replace("-REF", "").replace("-LE", "").strip()
        clean = alias_map.get(clean.upper(), clean)
        if clean != ch and clean not in raw.ch_names:
            rename_dict[ch] = clean
    if rename_dict:
        raw.rename_channels(rename_dict)
    return raw


def condition_array(
    data: np.ndarray,
    sfreq: float,
    target_sfreq: float = 256.0,
    l_freq: float = 1.0,
    h_freq: float = 80.0,
    notch_freqs: Tuple[float, ...] = (50.0, 60.0),
) -> np.ndarray:
    x = np.nan_to_num(data.astype(np.float64), nan=0.0)
    if np.std(x) < 1e-2:
        x = x * 1e6

    nyquist = sfreq / 2.0
    for f0 in notch_freqs:
        if f0 < nyquist - 1.0:
            b, a = signal.iirnotch(f0, Q=30.0, fs=sfreq)
            x = signal.filtfilt(b, a, x, axis=-1)

    hi_c = min(float(h_freq), nyquist - 1.5)
    if hi_c > l_freq + 0.5:
        sos = signal.butter(4, [float(l_freq), hi_c], btype="bandpass", fs=sfreq, output="sos")
        x = signal.sosfiltfilt(sos, x, axis=-1)

    if not math.isclose(sfreq, target_sfreq, rel_tol=1e-3):
        gcd = math.gcd(int(round(target_sfreq)), int(round(sfreq)))
        up = int(round(target_sfreq)) // gcd
        down = int(round(sfreq)) // gcd
        x = signal.resample_poly(x, up, down, axis=-1)

    return x.astype(np.float32)


def extract_event_locked_epochs(
    data: np.ndarray,
    sfreq: float,
    events_tsv: Path,
    win_sec: float = 4.0,
    stride_sec: float = 2.0,
    pre_probe_sec: float = 12.0,
) -> Tuple[List[Tuple[np.ndarray, int]], np.ndarray]:
    win_samples = int(win_sec * sfreq)
    stride_samples = int(stride_sec * sfreq)
    total_samples = data.shape[1]

    baseline_win = data[:, :win_samples].copy()

    if not events_tsv.exists():
        return [], baseline_win

    ev_df = pd.read_csv(events_tsv, sep="\t")
    trial_col = next((c for c in ["trial_type", "value", "response"] if c in ev_df.columns), None)
    if trial_col is None or "onset" not in ev_df.columns:
        return [], baseline_win

    epochs: List[Tuple[np.ndarray, int]] = []
    found_rest_baseline = False

    for _, row in ev_df.iterrows():
        desc = str(row[trial_col]).lower()
        if any(k in desc for k in ["rest", "baseline", "eyes_closed"]):
            label = 0
        elif any(k in desc for k in ["wandering", "distract", "mw", "off_task"]):
            label = 1
        elif any(k in desc for k in ["medit", "focus", "concentrat", "on_task"]):
            label = 2
        else:
            continue

        onset_s = int(float(row["onset"]) * sfreq)
        if label == 0 and not found_rest_baseline and onset_s + win_samples <= total_samples:
            cand = data[:, onset_s : onset_s + win_samples]
            if np.max(np.abs(cand)) < 250.0:
                baseline_win = cand.copy()
                found_rest_baseline = True

        start_s = max(0, onset_s - int(pre_probe_sec * sfreq))
        end_s = min(total_samples, onset_s)

        for pos in range(start_s, end_s - win_samples + 1, stride_samples):
            seg = data[:, pos : pos + win_samples]
            if np.max(np.abs(seg)) < 250.0:
                epochs.append((seg.astype(np.float32), label))

    return epochs, baseline_win.astype(np.float32)

Overwriting openmedwave/preprocessing.py


In [11]:
%%writefile openmedwave/features_gpu.py
import torch
import torch.nn as nn


class GPUNeuroPhysExtractor(nn.Module):
    def __init__(self, sfreq: float = 256.0, win_samples: int = 1024, num_frontal: int = 10, max_ch: int = 19):
        super().__init__()
        freqs = torch.fft.rfftfreq(win_samples, d=1.0 / sfreq)
        self.register_buffer("window", torch.hann_window(win_samples))

        bands = {
            "delta": (1.0, 4.0), "theta": (4.0, 8.0),
            "alpha1": (8.0, 10.0), "alpha2": (10.0, 12.0),
            "beta_low": (13.0, 20.0), "beta_high": (20.0, 30.0),
            "gamma": (30.0, 45.0),
        }
        for name, (lo, hi) in bands.items():
            self.register_buffer(f"m_{name}", (freqs >= lo) & (freqs < hi))

        self.register_buffer("m_total", (freqs >= 1.0) & (freqs <= 45.0))
        self.register_buffer(
            "m_aperiodic",
            ((freqs >= 2.0) & (freqs <= 4.0)) | ((freqs >= 14.0) & (freqs <= 40.0)),
        )
        log_f = torch.log10(freqs[self.m_aperiodic] + 1e-8)
        self.register_buffer("log_f_centered", log_f - log_f.mean())
        self.register_buffer("log_f_var", (self.log_f_centered ** 2).sum() + 1e-8)

        frontal_mask = torch.zeros(max_ch, dtype=torch.bool)
        frontal_mask[:num_frontal] = True
        self.register_buffer("frontal_mask", frontal_mask)

    def _extract_single(self, x: torch.Tensor, ch_ids: torch.Tensor, ch_mask: torch.Tensor) -> torch.Tensor:
        x_win = (x - x.mean(dim=-1, keepdim=True)) * self.window
        psd = (torch.fft.rfft(x_win, dim=-1).abs() ** 2) / x.size(-1)
        rel_psd = psd / (psd[..., self.m_total].sum(dim=-1, keepdim=True) + 1e-12)

        mask_f = ch_mask.float()
        valid_c = mask_f.sum(dim=-1, keepdim=True).clamp(min=1.0)

        def band_mean(b_mask):
            bp = rel_psd[..., b_mask].sum(dim=-1)
            return bp, (bp * mask_f).sum(dim=-1) / valid_c.squeeze(-1)

        _, delta = band_mean(self.m_delta)
        _, theta = band_mean(self.m_theta)
        a1_ch, a1 = band_mean(self.m_alpha1)
        a2_ch, a2 = band_mean(self.m_alpha2)
        _, bl = band_mean(self.m_beta_low)
        _, bh = band_mean(self.m_beta_high)
        _, gamma = band_mean(self.m_gamma)

        alpha_ch = a1_ch + a2_ch
        alpha = a1 + a2
        vigilance = (theta + alpha) / (delta + bh + 1e-6)
        theta_beta = theta / (bl + bh + 1e-6)

        is_frontal = self.frontal_mask[ch_ids] & ch_mask
        is_post = (~self.frontal_mask[ch_ids]) & ch_mask
        f_alpha = (alpha_ch * is_frontal.float()).sum(dim=-1) / is_frontal.float().sum(dim=-1).clamp(min=1.0)
        p_alpha = (alpha_ch * is_post.float()).sum(dim=-1) / is_post.float().sum(dim=-1).clamp(min=1.0)
        alpha_shift = f_alpha / (p_alpha + 1e-6)

        mean_psd = (psd * mask_f.unsqueeze(-1)).sum(dim=1) / valid_c
        log_p = torch.log10(mean_psd[:, self.m_aperiodic] + 1e-12)
        slope_1f = ((log_p - log_p.mean(dim=-1, keepdim=True)) * self.log_f_centered).sum(dim=-1) / self.log_f_var

        p_tot = rel_psd[..., self.m_total]
        p_tot = p_tot / (p_tot.sum(dim=-1, keepdim=True) + 1e-12)
        ent_ch = -(p_tot * torch.log2(p_tot + 1e-12)).sum(dim=-1) / torch.log2(
            torch.tensor(float(p_tot.size(-1)), device=x.device)
        )
        spec_entropy = (ent_ch * mask_f).sum(dim=-1) / valid_c.squeeze(-1)

        ll_ch = (x[..., 1:] - x[..., :-1]).abs().mean(dim=-1) / (x.std(dim=-1) + 1e-6)
        line_length = (ll_ch * mask_f).sum(dim=-1) / valid_c.squeeze(-1)

        return torch.stack(
            [
                torch.log10(delta + 1e-8), torch.log10(theta + 1e-8),
                torch.log10(a1 + 1e-8), torch.log10(a2 + 1e-8),
                torch.log10(bl + 1e-8), torch.log10(bh + 1e-8),
                torch.log10(gamma + 1e-8), vigilance, theta_beta,
                alpha_shift, slope_1f, spec_entropy, line_length,
            ],
            dim=-1,
        )

    def forward(
        self, eeg: torch.Tensor, baseline_eeg: torch.Tensor, ch_ids: torch.Tensor, ch_mask: torch.Tensor
    ) -> torch.Tensor:
        task_f = self._extract_single(eeg, ch_ids, ch_mask)
        base_f = self._extract_single(baseline_eeg, ch_ids, ch_mask)
        delta_f = task_f.clone()
        delta_f[:, :-1] = task_f[:, :-1] - base_f[:, :-1]
        return delta_f

Overwriting openmedwave/features_gpu.py


In [12]:
%%writefile openmedwave/model.py
from typing import Dict, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from .features_gpu import GPUNeuroPhysExtractor


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, alpha: float) -> torch.Tensor:
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return grad_output.neg() * ctx.alpha, None


class DynamicWaveletScaleFolder(nn.Module):
    def __init__(self, d_model: int = 128, patch_size: int = 64, n_levels: int = 4):
        super().__init__()
        self.n_levels = n_levels
        self.low_filters = nn.ModuleList([
            nn.Conv1d(1, 1, kernel_size=7, padding=3 * (2 ** lvl), dilation=2 ** lvl, bias=False)
            for lvl in range(n_levels)
        ])
        self.high_filters = nn.ModuleList([
            nn.Conv1d(1, 1, kernel_size=7, padding=3 * (2 ** lvl), dilation=2 ** lvl, bias=False)
            for lvl in range(n_levels)
        ])
        self.fold_weights = nn.Parameter(torch.ones(n_levels + 1) / (n_levels + 1))
        self.patch_proj = nn.Conv1d(1, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, t = x.shape
        approx = x.reshape(b * c, 1, t)
        subbands = []
        for lvl in range(self.n_levels):
            lo = self.low_filters[lvl](approx)[..., :t]
            hi = self.high_filters[lvl](approx)[..., :t]
            subbands.append(hi)
            approx = lo
        subbands.append(approx)

        weights = F.softmax(self.fold_weights, dim=0)
        folded = sum(w * sb for w, sb in zip(weights, subbands))
        tokens = self.patch_proj(folded).transpose(1, 2)
        return tokens.reshape(b, c, tokens.size(1), tokens.size(2))


class NeuroWaveMed(nn.Module):
    def __init__(
        self,
        num_canonical_channels: int = 19,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 3,
        num_states: int = 3,
        num_subjects: int = 50,
        sfreq: float = 256.0,
        win_samples: int = 1024,
        patch_size: int = 64,
        grl_alpha: float = 0.0,
    ):
        super().__init__()
        self.d_model = d_model
        self.grl_alpha = grl_alpha
        self.max_patches = max(1, win_samples // patch_size)

        self.neuro_extractor = GPUNeuroPhysExtractor(
            sfreq=sfreq, win_samples=win_samples, max_ch=num_canonical_channels
        )
        self.ch_embed = nn.Embedding(num_canonical_channels, d_model)
        self.time_embed = nn.Embedding(self.max_patches, d_model)
        self.wavelet_encoder = DynamicWaveletScaleFolder(
            d_model=d_model, patch_size=patch_size, n_levels=4
        )

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=0.2,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers, enable_nested_tensor=False)

        self.channel_attn = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1)
        )
        self.neuro_mlp = nn.Sequential(
            nn.LayerNorm(13), nn.Linear(13, 128), nn.GELU(), nn.Dropout(0.15), nn.Linear(128, d_model * 2)
        )
        self.fusion_norm = nn.LayerNorm(d_model)
        self.state_head = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128, num_states)
        )
        self.subject_discriminator = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Linear(128, num_subjects)
        )

    def forward(
        self,
        eeg: torch.Tensor,
        baseline_eeg: torch.Tensor,
        ch_ids: torch.Tensor,
        ch_mask: torch.Tensor,
        grl_alpha: Optional[float] = None,
    ) -> Dict[str, torch.Tensor]:
        b, c_max, _ = eeg.shape
        alpha = self.grl_alpha if grl_alpha is None else grl_alpha
        neuro_feats = self.neuro_extractor(eeg, baseline_eeg, ch_ids, ch_mask)

        eeg_norm = (eeg - eeg.mean(dim=-1, keepdim=True)) / (eeg.std(dim=-1, keepdim=True) + 1e-6)
        tokens = self.wavelet_encoder(eeg_norm)
        s = tokens.size(2)

        spatial_emb = self.ch_embed(ch_ids).unsqueeze(2)
        time_ids = torch.arange(s, device=eeg.device)
        temporal_emb = self.time_embed(time_ids).view(1, 1, s, self.d_model)

        tokens = (tokens + spatial_emb + temporal_emb).reshape(b, c_max * s, self.d_model)
        token_mask = ch_mask.unsqueeze(-1).expand(-1, -1, s).reshape(b, c_max * s)
        encoded = self.transformer(tokens, src_key_padding_mask=~token_mask)

        ch_repr = encoded.reshape(b, c_max, s, self.d_model).mean(dim=2)
        attn_scores = self.channel_attn(ch_repr).squeeze(-1).masked_fill(~ch_mask, -1e4)
        attn_weights = F.softmax(attn_scores, dim=-1).unsqueeze(-1)
        z_wavelet = torch.sum(ch_repr * attn_weights, dim=1)

        gamma, beta = torch.chunk(self.neuro_mlp(neuro_feats), 2, dim=-1)
        z_fused = self.fusion_norm(z_wavelet * (1.0 + gamma) + beta)

        z_rev = GradientReversalFunction.apply(z_fused, alpha)
        return {
            "state_logits": self.state_head(z_fused),
            "subject_logits": self.subject_discriminator(z_rev),
            "channel_weights": attn_weights.squeeze(-1),
        }

Overwriting openmedwave/model.py


In [13]:
import hashlib
import yaml
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold

cfg = yaml.safe_load(Path("configs/montages.yaml").read_text())
CANONICAL = cfg["canonical_channels"]
C_MAX = len(CANONICAL)
SFREQ = float(cfg["target_sfreq"])
WIN_SAMPLES = int(cfg["window_sec"] * SFREQ)

meta_path = Path(f"{MEMMAP_DIR}/meta.npz")
if not meta_path.exists():
    SHARD_DIR = Path(f"{PROJECT_DIR}/processed_shards")
    shard_files = sorted(SHARD_DIR.glob("*.npz"))
    assert len(shard_files) > 0, f"No .npz shards found in {SHARD_DIR}!"
    total_n = sum(len(np.load(sf)["labels"]) for sf in shard_files)
    mm_eeg = np.memmap(f"{MEMMAP_DIR}/eeg.dat", dtype="float32", mode="w+", shape=(total_n, C_MAX, WIN_SAMPLES))
    mm_base = np.memmap(f"{MEMMAP_DIR}/base.dat", dtype="float32", mode="w+", shape=(total_n, C_MAX, WIN_SAMPLES))
    avail_masks = np.zeros((total_n, C_MAX), dtype=bool)
    subject_ids = np.zeros(total_n, dtype=np.int64)
    state_labels = np.zeros(total_n, dtype=np.int64)
    cursor = 0
    for sf in shard_files:
        with np.load(sf) as d:
            n_k = len(d["labels"])
            mm_eeg[cursor : cursor + n_k] = d["eeg"]
            mm_base[cursor : cursor + n_k] = d["base"]
            avail_masks[cursor : cursor + n_k] = d["avail_mask"]
            subject_ids[cursor : cursor + n_k] = d["subject_id"]
            state_labels[cursor : cursor + n_k] = d["labels"]
            cursor += n_k
    mm_eeg.flush()
    mm_base.flush()
    np.savez(meta_path, n_samples=total_n, avail_masks=avail_masks, subject_ids=subject_ids, state_labels=state_labels)
    del mm_eeg, mm_base

meta = np.load(meta_path)
total_n = int(meta["n_samples"])
mm_eeg_r = np.memmap(f"{MEMMAP_DIR}/eeg.dat", dtype="float32", mode="r", shape=(total_n, C_MAX, WIN_SAMPLES))
valid_mask = np.ones(total_n, dtype=bool)
for start in range(0, total_n, 512):
    end = min(total_n, start + 512)
    x = np.array(mm_eeg_r[start:end], copy=True)
    m = meta["avail_masks"][start:end]
    lbls = meta["state_labels"][start:end]
    for i in range(end - start):
        active_x = x[i][m[i]]
        if lbls[i] == 0 or active_x.size == 0 or np.max(np.abs(active_x)) < 1e-4 or np.std(active_x) < 1e-5:
            valid_mask[start + i] = False
del mm_eeg_r

valid_indices = np.flatnonzero(valid_mask)
all_subjects = meta["subject_ids"][valid_indices]
binary_labels = meta["state_labels"][valid_indices] - 1


class MemmapMeditationDataset(Dataset):
    def __init__(
        self,
        memmap_dir: str,
        indices: np.ndarray,
        canonical_channels: list,
        wearable_montages: dict,
        is_train: bool = True,
        fixed_montage: str = None,
        channel_drop_prob: float = 0.20,
    ):
        meta_d = np.load(f"{memmap_dir}/meta.npz")
        n_total = int(meta_d["n_samples"])
        self.c_max = len(canonical_channels)
        self.ch_to_idx = {ch: i for i, ch in enumerate(canonical_channels)}
        self.wearable_montages = wearable_montages
        self.montage_names = list(wearable_montages.keys())
        self.indices = indices
        self.is_train = is_train
        self.fixed_montage = fixed_montage
        self.channel_drop_prob = channel_drop_prob

        self.mm_eeg = np.memmap(f"{memmap_dir}/eeg.dat", dtype="float32", mode="r", shape=(n_total, self.c_max, WIN_SAMPLES))
        self.mm_base = np.memmap(f"{memmap_dir}/base.dat", dtype="float32", mode="r", shape=(n_total, self.c_max, WIN_SAMPLES))
        self.avail_masks = meta_d["avail_masks"]
        self.subject_ids = meta_d["subject_ids"]
        self.state_labels = meta_d["state_labels"]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = int(self.indices[i])
        eeg = np.array(self.mm_eeg[idx], copy=True)
        base = np.array(self.mm_base[idx], copy=True)
        avail_mask = self.avail_masks[idx]

        if self.fixed_montage is not None:
            target_chs = list(self.wearable_montages[self.fixed_montage])
        elif self.is_train:
            target_chs = list(self.wearable_montages[random.choice(self.montage_names)])
            if len(target_chs) > 2 and random.random() < self.channel_drop_prob:
                target_chs.pop(random.randrange(len(target_chs)))
        else:
            target_chs = [ch for ch, ok in zip(CANONICAL, avail_mask) if ok]

        ch_mask = np.zeros(self.c_max, dtype=bool)
        for ch in target_chs:
            slot = self.ch_to_idx[ch]
            if avail_mask[slot]:
                ch_mask[slot] = True
        if not ch_mask.any():
            ch_mask = avail_mask.copy()

        eeg[~ch_mask] = 0.0
        base[~ch_mask] = 0.0

        if self.is_train:
            eeg = eeg * np.random.uniform(0.85, 1.15, size=(self.c_max, 1)).astype(np.float32)
            if random.random() < 0.30:
                t0 = random.randint(0, WIN_SAMPLES - 128)
                eeg[:, t0 : t0 + 128] = 0.0

        return {
            "eeg": torch.from_numpy(eeg),
            "baseline_eeg": torch.from_numpy(base),
            "ch_ids": torch.arange(self.c_max, dtype=torch.long),
            "ch_mask": torch.from_numpy(ch_mask),
            "subject_id": torch.tensor(self.subject_ids[idx], dtype=torch.long),
            "state_label": torch.tensor(self.state_labels[idx] - 1, dtype=torch.long),
        }

print(f"Active 2-class epochs: {len(valid_indices)}/{total_n} across {len(np.unique(all_subjects))} subjects")

Active 2-class epochs: 3828/4010 across 24 subjects


In [14]:
import gc
import json
import pickle
from datetime import datetime
import pandas as pd
from scipy.signal import butter, sosfiltfilt, welch
from scipy.linalg import logm
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = Path(f"{BENCH_DIR}/run_{RUN_TAG}")
LATEST_DIR = Path(BENCH_LATEST_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LATEST_DIR.mkdir(parents=True, exist_ok=True)

full_ds = MemmapMeditationDataset(
    MEMMAP_DIR, valid_indices, CANONICAL, cfg["wearable_montages"],
    is_train=False, fixed_montage="muse2_4ch",
)
full_loader = DataLoader(full_ds, batch_size=128, shuffle=False, num_workers=2)

muse_chunks, y_chunks, subj_chunks = [], [], []
for batch in full_loader:
    eeg_b = batch["eeg"].numpy()
    mask_b = batch["ch_mask"].numpy().astype(bool)
    active_4ch = np.stack([eeg_b[i][mask_b[i]] for i in range(len(eeg_b))], axis=0)
    muse_chunks.append(active_4ch)
    y_chunks.append(batch["state_label"].numpy())
    subj_chunks.append(batch["subject_id"].numpy())

X_muse_raw = np.concatenate(muse_chunks, axis=0).astype(np.float32)
y_all = np.concatenate(y_chunks, axis=0)
subj_all = np.concatenate(subj_chunks, axis=0)

FB_BANDS_HF = [
    (1, 4), (4, 8), (8, 10), (10, 13),
    (13, 20), (20, 30), (30, 45), (45, 80),
    (4, 13), (1, 80),
]


def extract_hf_riemannian(eeg_batch, sfreq=SFREQ, trace_norm=False, include_rel_psd=False):
    N, C, T = eeg_batch.shape
    tri_idx = np.triu_indices(C)
    nyq = sfreq / 2.0
    feats = []

    for lo, hi in FB_BANDS_HF:
        hi_c = min(hi, nyq - 2.0)
        sos = butter(4, [lo, hi_c], btype="bandpass", fs=sfreq, output="sos")
        filt = sosfiltfilt(sos, eeg_batch, axis=-1)
        band_f = np.zeros((N, len(tri_idx[0])), dtype=np.float32)
        for i in range(N):
            cov = np.cov(filt[i])
            if trace_norm:
                cov = cov / (np.trace(cov) + 1e-8)
            cov = cov + 1e-5 * np.eye(C)
            band_f[i] = np.real(logm(cov))[tri_idx]
        feats.append(band_f)

    d1 = np.diff(eeg_batch, axis=-1)
    d2 = np.diff(d1, axis=-1)
    var0 = np.var(eeg_batch, axis=-1) + 1e-8
    var1 = np.var(d1, axis=-1) + 1e-8
    var2 = np.var(d2, axis=-1) + 1e-8
    mob = np.sqrt(var1 / var0)
    comp = np.sqrt(var2 / var1) / (mob + 1e-8)
    if not trace_norm:
        feats.append(np.log(var0).astype(np.float32))
    feats.extend([mob.astype(np.float32), comp.astype(np.float32)])

    freqs, psd = welch(eeg_batch, fs=sfreq, nperseg=min(256, T), axis=-1)
    f_mask_tot = (freqs >= 1.0) & (freqs <= min(80.0, nyq - 2.0))
    psd_tot = psd[..., f_mask_tot]
    tot_p = np.sum(psd_tot, axis=-1, keepdims=True) + 1e-12
    psd_norm = psd_tot / tot_p
    spec_ent = -np.sum(psd_norm * np.log2(psd_norm + 1e-12), axis=-1)
    feats.append(spec_ent.astype(np.float32))

    if include_rel_psd:
        for f_lo, f_hi in [(1, 4), (4, 8), (8, 13), (13, 30), (30, 45), (45, 80)]:
            bm = (freqs >= f_lo) & (freqs <= min(f_hi, nyq - 2.0))
            rel_p = np.sum(psd[..., bm], axis=-1) / tot_p.squeeze(-1)
            feats.append(np.log(rel_p + 1e-8).astype(np.float32))
    else:
        def _bp(f_lo, f_hi):
            m = (freqs >= f_lo) & (freqs <= min(f_hi, nyq - 2.0))
            return np.mean(psd[..., m], axis=-1) + 1e-12
        p_t, p_a = _bp(4, 8), _bp(8, 13)
        p_b, p_g = _bp(13, 30), _bp(30, 45)
        feats.extend([
            np.log(p_t / p_b).astype(np.float32),
            np.log(p_a / p_t).astype(np.float32),
            np.log(p_g / p_a).astype(np.float32),
        ])

    return np.concatenate(feats, axis=1).astype(np.float32)


def eval_stratified_lda(X_feat, cal_frac=0.20):
    accs, bas, aucs = [], [], []
    for subj in np.unique(subj_all):
        m = (subj_all == subj)
        X_s, y_s = X_feat[m], y_all[m]
        if len(np.unique(y_s)) < 2 or len(y_s) < 15 or np.min(np.bincount(y_s)) < 3:
            continue
        sss = StratifiedShuffleSplit(n_splits=1, train_size=cal_frac, random_state=42)
        cal_pos, test_pos = next(sss.split(X_s, y_s))
        clf = make_pipeline(
            StandardScaler(),
            LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto", priors=[0.5, 0.5]),
        )
        clf.fit(X_s[cal_pos], y_s[cal_pos])
        probs = clf.predict_proba(X_s[test_pos])[:, 1]
        preds = (probs >= 0.5).astype(int)
        accs.append(accuracy_score(y_s[test_pos], preds))
        bas.append(balanced_accuracy_score(y_s[test_pos], preds))
        aucs.append(roc_auc_score(y_s[test_pos], probs))
    return float(np.mean(accs) * 100), float(np.mean(bas) * 100), float(np.mean(aucs))


def eval_chrono_blend(X_feat, cal_frac=0.30):
    accs, bas, aucs = [], [], []
    for subj in np.unique(subj_all):
        m = (subj_all == subj)
        X_s, y_s = X_feat[m], y_all[m]
        n = len(y_s)
        if len(np.unique(y_s)) < 2 or n < 20:
            continue
        sp = int(n * cal_frac)
        if len(np.unique(y_s[:sp])) == 2 and len(np.unique(y_s[sp:])) == 2:
            X_cal, y_cal, X_te, y_te = X_s[:sp], y_s[:sp], X_s[sp:], y_s[sp:]
        elif len(np.unique(y_s[-sp:])) == 2 and len(np.unique(y_s[:-sp])) == 2:
            X_cal, y_cal, X_te, y_te = X_s[-sp:], y_s[-sp:], X_s[:-sp], y_s[:-sp:]
        else:
            continue
        lda = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto", priors=[0.5, 0.5]))
        lr  = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, class_weight="balanced", max_iter=1000, random_state=42))
        lda.fit(X_cal, y_cal)
        lr.fit(X_cal, y_cal)
        probs = 0.5 * lda.predict_proba(X_te)[:, 1] + 0.5 * lr.predict_proba(X_te)[:, 1]
        preds = (probs >= 0.5).astype(int)
        accs.append(accuracy_score(y_te, preds))
        bas.append(balanced_accuracy_score(y_te, preds))
        aucs.append(roc_auc_score(y_te, probs))
    return float(np.mean(accs) * 100), float(np.mean(bas) * 100), float(np.mean(aucs))


X_plus_hf = extract_hf_riemannian(X_muse_raw, trace_norm=False, include_rel_psd=False)
X_inv = extract_hf_riemannian(X_muse_raw, trace_norm=True, include_rel_psd=True)

hf20_acc, hf20_ba, hf20_auc = eval_stratified_lda(X_plus_hf, cal_frac=0.20)
hf30_acc, hf30_ba, hf30_auc = eval_stratified_lda(X_plus_hf, cal_frac=0.30)
hfc_acc,  hfc_ba,  hfc_auc  = eval_chrono_blend(X_plus_hf, cal_frac=0.30)
invc_acc, invc_ba, invc_auc = eval_chrono_blend(X_inv,     cal_frac=0.30)

benchmark_rows = [
    {"Pipeline": "Zero-Shot NeuroWaveMed v4", "Split / Cal": "5-Fold GroupKFold (0% Cal)", "Raw_Acc_%": 58.89, "Bal_Acc_%": 57.15, "ROC_AUC": 0.5599},
    {"Pipeline": "AdamW Fine-Tuning (v4)", "Split / Cal": "5-Fold Stratified (20% Cal)", "Raw_Acc_%": 66.78, "Bal_Acc_%": 64.78, "ROC_AUC": 0.6000},
    {"Pipeline": "Frozen v4 Embeddings + RBF-SVM", "Split / Cal": "5-Fold Stratified (20% Cal)", "Raw_Acc_%": 68.96, "Bal_Acc_%": 65.83, "ROC_AUC": 0.6821},
    {"Pipeline": "5-Band Riemannian (1-30 Hz) + Shrinkage LDA", "Split / Cal": "Stratified (20% Cal)", "Raw_Acc_%": 71.08, "Bal_Acc_%": 65.74, "ROC_AUC": 0.7060},
    {"Pipeline": "9-Band Filter-Bank + Gamma (1-45 Hz) + Hjorth", "Split / Cal": "Stratified (20% / 30% Cal)", "Raw_Acc_%": 74.51, "Bal_Acc_%": 68.17, "ROC_AUC": 0.7580},
    {"Pipeline": "1-80 Hz High-Gamma Riemannian + Shrinkage LDA", "Split / Cal": "Stratified (20% Cal)", "Raw_Acc_%": round(hf20_acc, 2), "Bal_Acc_%": round(hf20_ba, 2), "ROC_AUC": round(hf20_auc, 4)},
    {"Pipeline": "1-80 Hz High-Gamma Riemannian + Shrinkage LDA", "Split / Cal": "Stratified (30% Cal)", "Raw_Acc_%": round(hf30_acc, 2), "Bal_Acc_%": round(hf30_ba, 2), "ROC_AUC": round(hf30_auc, 4)},
    {"Pipeline": "1-80 Hz High-Gamma (Static LDA+LR Blend)", "Split / Cal": "Strict Chronological (30% Cal)", "Raw_Acc_%": round(hfc_acc, 2), "Bal_Acc_%": round(hfc_ba, 2), "ROC_AUC": round(hfc_auc, 4)},
    {"Pipeline": "Amplitude-Invariant 1-80 Hz (Trace-Norm + Rel PSD)", "Split / Cal": "Strict Chronological (30% Cal)", "Raw_Acc_%": round(invc_acc, 2), "Bal_Acc_%": round(invc_ba, 2), "ROC_AUC": round(invc_auc, 4)},
]

df_benchmark = pd.DataFrame(benchmark_rows)
print(df_benchmark.to_string(index=False))


def _save_outputs(target_dir: Path):
    df_benchmark.to_csv(target_dir / "benchmark_summary.csv", index=False)
    df_benchmark.to_json(target_dir / "benchmark_summary.json", orient="records", indent=2)
    np.savez_compressed(
        target_dir / "features_and_labels.npz",
        valid_indices=valid_indices,
        y_all=y_all,
        subj_all=subj_all,
        X_plus_hf=X_plus_hf,
        X_inv=X_inv,
    )
    with open(target_dir / "checkpoint_bundle.pkl", "wb") as f:
        pickle.dump({"saved_at": datetime.now().isoformat(), "sfreq": float(SFREQ)}, f)
    (target_dir / "manifest.json").write_text(
        json.dumps({"run_tag": RUN_TAG, "saved_at": datetime.now().isoformat()}, indent=2)
    )


_save_outputs(SAVE_DIR)
_save_outputs(LATEST_DIR)
print(f"\nCheckpointed to {SAVE_DIR} and {LATEST_DIR}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 2.8511703043017915e-13
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 2.368681878823296e-13
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 4.3397554348178723e-13
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 2.8624029898505804e-13
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 3.001400168565572e-13
  return 

                                          Pipeline                    Split / Cal  Raw_Acc_%  Bal_Acc_%  ROC_AUC
                         Zero-Shot NeuroWaveMed v4     5-Fold GroupKFold (0% Cal)      58.89      57.15   0.5599
                            AdamW Fine-Tuning (v4)    5-Fold Stratified (20% Cal)      66.78      64.78   0.6000
                    Frozen v4 Embeddings + RBF-SVM    5-Fold Stratified (20% Cal)      68.96      65.83   0.6821
       5-Band Riemannian (1-30 Hz) + Shrinkage LDA           Stratified (20% Cal)      71.08      65.74   0.7060
     9-Band Filter-Bank + Gamma (1-45 Hz) + Hjorth     Stratified (20% / 30% Cal)      74.51      68.17   0.7580
     1-80 Hz High-Gamma Riemannian + Shrinkage LDA           Stratified (20% Cal)      75.01      68.27   0.7496
     1-80 Hz High-Gamma Riemannian + Shrinkage LDA           Stratified (30% Cal)      76.35      70.24   0.7686
          1-80 Hz High-Gamma (Static LDA+LR Blend) Strict Chronological (30% Cal)      61.38    

In [15]:
import matplotlib.pyplot as plt
import gradio as gr

npz_path = Path(BENCH_LATEST_DIR) / "features_and_labels.npz"
assert npz_path.exists(), f"Missing {npz_path}"
npz = np.load(npz_path, allow_pickle=True)
X_ckpt = npz["X_plus_hf"].astype(np.float32)
y_ckpt = npz["y_all"].astype(int)
subj_ckpt = npz["subj_all"].astype(int)

valid_subjects = [
    int(s) for s in sorted(np.unique(subj_ckpt))
    if len(np.unique(y_ckpt[subj_ckpt == s])) >= 2
    and len(y_ckpt[subj_ckpt == s]) >= 15
    and np.min(np.bincount(y_ckpt[subj_ckpt == s])) >= 3
]


def _plot_timeline(x_sec, probs, smooth_win, title, win_sec=4.0, truth=None):
    fig, ax = plt.subplots(figsize=(10.0, 4.0), dpi=115)
    if truth is not None:
        for i in range(len(truth)):
            x0, x1 = x_sec[i] - win_sec / 2.0, x_sec[i] + win_sec / 2.0
            ax.axvspan(x0, x1, color=("#2ecc71" if truth[i] == 1 else "#e67e22"), alpha=0.12, lw=0)
    ax.plot(x_sec, probs, marker="o", markersize=3, linewidth=1.6, label="P(Focused Meditation)")
    ax.axhline(0.50, linestyle="--", color="red", linewidth=1.1, label="Threshold (0.50)")
    ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel("Elapsed Time (s)")
    ax.set_ylabel("Focused Meditation Probability")
    ax.set_title(title)
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.legend(loc="upper right")
    fig.tight_layout()
    return fig


def run_subject_tab(sid, cal_pct, smooth_win, show_truth):
    m = (subj_ckpt == int(sid))
    X_s, y_s = X_ckpt[m], y_ckpt[m]
    cal_i, test_i = next(StratifiedShuffleSplit(n_splits=1, train_size=float(cal_pct) / 100.0, random_state=42).split(X_s, y_s))
    clf = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto", priors=[0.5, 0.5]))
    clf.fit(X_s[cal_i], y_s[cal_i])
    raw_p = clf.predict_proba(X_s[test_i])[:, 1]
    probs = pd.Series(raw_p).rolling(window=int(smooth_win), min_periods=1).mean().values if int(smooth_win) > 1 else raw_p
    preds = (probs >= 0.5).astype(int)
    y_te = y_s[test_i]
    x_sec = np.arange(len(probs)) * 4.0
    title = (
        f"Subject {sid} | Acc: {accuracy_score(y_te, preds)*100:.2f}% | "
        f"Bal-Acc: {balanced_accuracy_score(y_te, preds)*100:.2f}% | AUC: {roc_auc_score(y_te, probs):.4f}"
    )
    return _plot_timeline(x_sec, probs, smooth_win, title, win_sec=4.0, truth=(y_te if show_truth else None))


def run_csv_tab(csv_file, sfreq, win_sec, cal_pct, smooth_win):
    if csv_file is None:
        raise ValueError("Please upload a 4-channel EEG CSV file.")
    df = pd.read_csv(csv_file.name if hasattr(csv_file, "name") else csv_file)
    num_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c])
        and c.lower() not in {"time", "timestamp", "timestamp_sec", "label", "state"}
    ][:4]
    assert len(num_cols) == 4, f"Expected 4 numeric EEG columns, found {num_cols}"
    arr = df[num_cols].dropna().to_numpy(dtype=np.float64)
    ws = max(32, int(round(float(sfreq) * float(win_sec))))
    nw = len(arr) // ws
    assert nw >= 6, f"Need at least 6 windows, got {nw}"
    wins = arr[: nw * ws].reshape(nw, ws, 4).transpose(0, 2, 1)
    Xw = extract_hf_riemannian(wins, sfreq=float(sfreq))
    cal_end = max(3, min(nw - 2, int(round(nw * float(cal_pct) / 100.0))))
    z_cal = StandardScaler().fit_transform(Xw[:cal_end])
    pc1 = np.linalg.svd(z_cal, full_matrices=False)[2][0]
    y_p = ((z_cal @ pc1) >= np.median(z_cal @ pc1)).astype(int)
    clf = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto", priors=[0.5, 0.5]))
    clf.fit(Xw[:cal_end], y_p)
    ev = np.arange(cal_end, nw)
    raw_p = clf.predict_proba(Xw[ev])[:, 1]
    probs = pd.Series(raw_p).rolling(window=int(smooth_win), min_periods=1).mean().values if int(smooth_win) > 1 else raw_p
    x_sec = ev * float(win_sec)
    return _plot_timeline(x_sec, probs, smooth_win, f"Uploaded 4-Ch CSV ({num_cols}) | Cal={cal_end} win -> Eval={len(ev)} win", win_sec=float(win_sec))


with gr.Blocks(title="NeuroWaveMed 4-Ch Riemannian Timeline App") as demo:
    gr.Markdown("# Focused Meditation Probability Timeline (1–80 Hz Riemannian + Shrinkage LDA)")
    with gr.Tabs():
        with gr.TabItem("Checkpointed Subjects"):
            s_dd = gr.Dropdown(choices=valid_subjects, value=valid_subjects[0], label="Subject ID")
            c_sl = gr.Slider(10, 50, step=5, value=30, label="Calibration %")
            sm_sl = gr.Slider(1, 7, step=1, value=1, label="Rolling Window Smoothing")
            tr_chk = gr.Checkbox(value=True, label="Show ground-truth shading")
            b1 = gr.Button("Plot Timeline", variant="primary")
            p1 = gr.Plot()
            b1.click(run_subject_tab, [s_dd, c_sl, sm_sl, tr_chk], p1)
            demo.load(run_subject_tab, [s_dd, c_sl, sm_sl, tr_chk], p1)
        with gr.TabItem("Upload 4-Ch CSV"):
            f_in = gr.File(label="4-Channel EEG CSV", file_types=[".csv"])
            sf_n = gr.Number(value=256.0, label="Sampling Rate (Hz)")
            ws_n = gr.Number(value=4.0, label="Window Duration (s)")
            c2_sl = gr.Slider(10, 50, step=5, value=30, label="Calibration %")
            sm2_sl = gr.Slider(1, 7, step=1, value=1, label="Rolling Window Smoothing")
            b2 = gr.Button("Calibrate & Plot CSV Timeline", variant="primary")
            p2 = gr.Plot()
            b2.click(run_csv_tab, [f_in, sf_n, ws_n, c2_sl, sm2_sl], p2)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://08812b413eddb308f9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
import gc
import json
import pickle
import shutil
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# ==============================================================================
# 1. ENSURE DRIVE MOUNT & TARGET CHECKPOINT DIRECTORIES
# ==============================================================================
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

BENCH_BASE = Path("/content/drive/MyDrive/NeuroWaveMed_Benchmarks")
OPENMEDWAVE_DIR = Path("/content/drive/MyDrive/OpenMedWave")
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = BENCH_BASE / f"run_{RUN_TAG}"
LATEST_DIR = BENCH_BASE / "latest"
NB_BACKUP_DIR = OPENMEDWAVE_DIR / "notebooks"

for d in [RUN_DIR, LATEST_DIR, NB_BACKUP_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def _write_checkpoint_to(target_dir: Path, g: dict) -> dict:
    saved_items = {}

    # A. Save feature & label arrays if present in notebook globals
    npz_payload = {}
    for key in [
        "valid_indices",
        "y_all",
        "subj_all",
        "probe_ids_all",
        "X_muse_raw",
        "X_plus_hf",
        "X_rich",
        "X_inv",
    ]:
        if key in g and g[key] is not None:
            npz_payload[key] = np.asarray(g[key])

    if npz_payload:
        npz_file = target_dir / "features_and_labels.npz"
        np.savez_compressed(npz_file, **npz_payload)
        saved_items["features_and_labels.npz"] = list(npz_payload.keys())

    # B. Save benchmark summary DataFrame / list-of-dicts if present
    df_bench = None
    for df_key in ["df_benchmark", "df_bench", "benchmark_rows"]:
        if df_key in g and g[df_key] is not None:
            val = g[df_key]
            if isinstance(val, pd.DataFrame):
                df_bench = val.copy()
                break
            if isinstance(val, list) and len(val) > 0:
                df_bench = pd.DataFrame(val)
                break

    if df_bench is not None and not df_bench.empty:
        df_bench.to_csv(target_dir / "benchmark_summary.csv", index=False)
        df_bench.to_json(target_dir / "benchmark_summary.json", orient="records", indent=2)
        saved_items["benchmark_summary.csv"] = len(df_bench)

    # C. Save any per-subject DataFrames present
    for sub_key, out_name in [
        ("df_subjs", "per_subject_probe_eval.csv"),
        ("df_probes", "per_probe_predictions.csv"),
        ("df_subjects", "per_subject_breakdown.csv"),
    ]:
        if sub_key in g and isinstance(g[sub_key], pd.DataFrame) and not g[sub_key].empty:
            g[sub_key].to_csv(target_dir / out_name, index=False)
            saved_items[out_name] = len(g[sub_key])

    # D. Save PyTorch model weights if present
    for model_key in ["model", "fold_model", "hybrid_model"]:
        if model_key in g and hasattr(g[model_key], "state_dict"):
            pt_file = target_dir / f"{model_key}_state.pt"
            torch.save(g[model_key].state_dict(), pt_file)
            saved_items[pt_file.name] = True

    # E. Save lightweight metadata bundle & manifest
    bundle = {
        "run_tag": RUN_TAG,
        "saved_at": datetime.now().isoformat(),
        "sfreq": float(g["SFREQ"]) if "SFREQ" in g else 256.0,
        "available_vars": sorted([k for k in npz_payload.keys()]),
    }
    with (target_dir / "checkpoint_bundle.pkl").open("wb") as f:
        pickle.dump(bundle, f)

    (target_dir / "manifest.json").write_text(
        json.dumps(
            {
                "run_tag": RUN_TAG,
                "saved_at": datetime.now().isoformat(),
                "target_dir": str(target_dir),
                "saved_items": saved_items,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    return saved_items


# ==============================================================================
# 2. BACK UP ANY .IPYNB FILES FOUND IN /CONTENT OR PROJECT DIR
# ==============================================================================
backed_up_nbs = []
for search_root in [Path("/content"), OPENMEDWAVE_DIR]:
    if not search_root.exists():
        continue
    for nb_file in search_root.rglob("*.ipynb"):
        if ".ipynb_checkpoints" in nb_file.parts or "drive/MyDrive/OpenMedWave/notebooks" in str(nb_file):
            continue
        dest_nb = NB_BACKUP_DIR / f"{nb_file.stem}_{RUN_TAG}.ipynb"
        try:
            shutil.copy2(nb_file, dest_nb)
            backed_up_nbs.append(str(dest_nb))
        except Exception:
            pass

# ==============================================================================
# 3. EXECUTE CHECKPOINT TO RUN FOLDER + LATEST MIRROR
# ==============================================================================
run_saved = _write_checkpoint_to(RUN_DIR, globals())
latest_saved = _write_checkpoint_to(LATEST_DIR, globals())

print(f"Checkpoint complete!")
print(f"  Run dir   : {RUN_DIR}")
print(f"  Latest dir: {LATEST_DIR}")
print(f"  Artifacts : {list(latest_saved.keys())}")
if backed_up_nbs:
    print(f"  Backed-up notebooks: {backed_up_nbs}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Checkpoint complete!
  Run dir   : /content/drive/MyDrive/NeuroWaveMed_Benchmarks/run_20261120_171848
  Latest dir: /content/drive/MyDrive/NeuroWaveMed_Benchmarks/latest
  Artifacts : ['features_and_labels.npz', 'benchmark_summary.csv']
  Backed-up notebooks: ['/content/drive/MyDrive/OpenMedWave/notebooks/00pytorchfundamentals_20261120_171848.ipynb', '/content/drive/MyDrive/OpenMedWave/notebooks/Untitled0_20261120_171848.ipynb', '/content/drive/MyDrive/OpenMedWave/notebooks/practiceai_20261120_171848.ipynb', '/content/drive/MyDrive/OpenMedWave/notebooks/Alz_20261120_171848.ipynb', '/content/drive/MyDrive/OpenMedWave/notebooks/alz1_20261120_171848.ipynb', '/content/drive/MyDrive/OpenMedWave/notebooks/Untitled1_20261120_171848.ipynb', '/content/drive/MyDrive/OpenMedWave/notebooks/eeg_20261120_171848.ipynb', '/content/drive/MyDrive/OpenMedWave/notebooks/Untitled2_20261120_171848.ipynb']
